In [0]:
# STEP 1: Load dataset into Delta Table

# READ CSV
df_master_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_master.csv")
)

display(df_master_raw)

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
8,Kiran Rao,kiran.rao@example.com,Hyderabad,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
10,Meena Iyer,meena.iyer@example.com,Coimbatore,active,2026-01-01


In [0]:
# Saving as Delta Table
(
    df_master_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customer_delta")
)

display(spark.table("customer_delta"))

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
8,Kiran Rao,kiran.rao@example.com,Hyderabad,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
10,Meena Iyer,meena.iyer@example.com,Coimbatore,active,2026-01-01


In [0]:
# STEP 2: Data Cleaning - Remove duplicates and handle nulls

df_master_clean = (
    df_master_raw
    .dropDuplicates(["customer_id"])
    .na.drop(subset=["email"])
)

display(df_master_clean)
print("Row count before cleaning:", df_master_raw.count())
print("Row count after cleaning:", df_master_clean.count())

customer_id,name,email,city,status,updated_at
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
10,Meena Iyer,meena.iyer@example.com,Coimbatore,active,2026-01-01
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
13,Arjun Nair,arjun.nair@example.com,Kochi,inactive,2026-01-01
14,Sneha Kapoor,sneha.kapoor@example.com,Jaipur,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01


Row count before cleaning: 18
Row count after cleaning: 14


In [0]:
(
    df_master_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("customer_delta")
)

display(spark.table("customer_delta"))

customer_id,name,email,city,status,updated_at
5,Amit Joshi,amit.joshi@example.com,Pune,active,2026-01-01
10,Meena Iyer,meena.iyer@example.com,Coimbatore,active,2026-01-01
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Delhi,active,2026-01-01
13,Arjun Nair,arjun.nair@example.com,Kochi,inactive,2026-01-01
14,Sneha Kapoor,sneha.kapoor@example.com,Jaipur,active,2026-01-01
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01


In [0]:
# STEP 3: Load incremental dataset

df_incremental = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/dataset/customer_incremental.csv")  # your exact path
)

display(df_incremental)
print("Incremental rows:", df_incremental.count())

customer_id,name,email,city,status,updated_at
2,Anjali Verma,anjali.verma@example.com,Gurgaon,active,2026-02-15
5,Amit Joshi,amit.joshi@example.com,Pune,inactive,2026-02-15
8,Kiran Rao,kiran.rao@example.com,Hyderabad,inactive,2026-02-15
13,Arjun Nair,arjun.nair@example.com,Kochi,active,2026-02-15
14,Sneha Kapoor,sneha.kapoor@example.com,Mumbai,active,2026-02-15
17,Ritu Bhatia,ritu.bhatia@example.com,Chandigarh,active,2026-02-15
18,Manoj Tiwari,manoj.tiwari@example.com,Bhopal,active,2026-02-15
19,Farah Khan,farah.khan@example.com,Nagpur,active,2026-02-15


Incremental rows: 8


In [0]:
df_incremental.createOrReplaceTempView("incremental_view")

In [0]:
%sql
MERGE INTO customer_delta AS target
USING incremental_view AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
UPDATE SET
  target.name = source.name,
  target.email = source.email,
  target.city = source.city,
  target.status = source.status,
  target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
INSERT (customer_id, name, email, city, status, updated_at)
VALUES (source.customer_id, source.name, source.email, source.city, source.status, source.updated_at)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
8,5,0,3


In [0]:
# STEP 5: Validation

print("Total row count after merge:", spark.table("customer_delta").count())

# check for duplicate customer_ids
dupes = spark.table("customer_delta").groupBy("customer_id").count().filter("count > 1")
display(dupes)
print("Duplicate customer_ids found:", dupes.count())

Total row count after merge: 17


customer_id,count


Duplicate customer_ids found: 0


In [0]:
# STEP 6: Final Output

display(spark.table("customer_delta").orderBy("customer_id"))

customer_id,name,email,city,status,updated_at
1,Ravi Sharma,ravi.sharma@example.com,Jaipur,active,2026-01-01
2,Anjali Verma,anjali.verma@example.com,Gurgaon,active,2026-02-15
3,Suresh Kumar,suresh.kumar@example.com,Mumbai,active,2026-01-01
4,Priya Singh,priya.singh@example.com,Bangalore,active,2026-01-01
5,Amit Joshi,amit.joshi@example.com,Pune,inactive,2026-02-15
6,Neha Gupta,neha.gupta@example.com,Delhi,active,2026-01-01
7,Rahul Mehta,rahul.mehta@example.com,Chennai,active,2026-01-01
8,Kiran Rao,kiran.rao@example.com,Hyderabad,inactive,2026-02-15
9,Deepak Nair,deepak.nair@example.com,Kochi,active,2026-01-01
10,Meena Iyer,meena.iyer@example.com,Coimbatore,active,2026-01-01


In [0]:

%sql
DESCRIBE HISTORY customer_delta

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-07-05T12:52:46.000Z,70395981917217,2023pietcslakshita098@poornima.org,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3926725108493679),df6af503-39a5-4de7-b116-4533900f79ec,0705-123329-bxak9jhd-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4917, p25FileSize -> 2601, numDeletionVectorsRemoved -> 1, minFileSize -> 2601, numAddedFiles -> 1, maxFileSize -> 2601, p75FileSize -> 2601, p50FileSize -> 2601, numAddedBytes -> 2601)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
2,2026-07-05T12:52:43.000Z,70395981917217,2023pietcslakshita098@poornima.org,MERGE,"Map(predicate -> [""(customer_id#12176 = customer_id#12163)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3926725108493679),df6af503-39a5-4de7-b116-4533900f79ec,0705-123329-bxak9jhd-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2333, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 6257, materializeSourceTimeMs -> 359, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2916, numTargetRowsUpdated -> 5, numOutputRows -> 8, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2877)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
1,2026-07-05T12:49:05.000Z,70395981917217,2023pietcslakshita098@poornima.org,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3926725108493679),bd218f74-6470-4d8c-983b-d3d5c3bfc4dc,0705-123329-bxak9jhd-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2711, numDeletionVectorsRemoved -> 0, numOutputRows -> 14, numOutputBytes -> 2584)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-07-05T12:41:42.000Z,70395981917217,2023pietcslakshita098@poornima.org,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3926725108493679),0a6327b2-7de3-4795-80b1-d8e44b333e2f,0705-123329-bxak9jhd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 18, numOutputBytes -> 2711)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13


### Summary

In this assignment I worked with Delta Lake to handle incremental data updates.

1. **Load** - Loaded the customer_master.csv into a dataframe and saved it as a Delta table (customer_delta).
2. **Clean** - Removed duplicate customer_id rows and dropped rows where email was missing. Row count went from 18 to 14 after cleaning.
3. **Incremental Data** - Loaded a second file (customer_incremental.csv) which simulates new data coming in later - some existing customers had updated details, and a few were completely new customers.
4. **Merge** - Used SQL MERGE INTO to update the matching customers and insert the new ones in a single operation. This updated 5 existing rows and inserted 3 new ones.
5. **Validation** - Checked the final row count (17) and confirmed there were no duplicate customer_id values left.
6. **Final Output** - Displayed the updated customer_delta table and also checked DESCRIBE HISTORY to see the different versions of the table.

**What I learned:** Delta Lake's MERGE is really useful for this kind of upsert scenario (update + insert together) instead of writing separate logic to check what exists and what's new. It also keeps a version history automatically which could be useful if I ever need to look back at older data.